# 07_06_指示の書き方を変えて正解率を測る

コード：`ch07/blocks/07_06_指示の書き方を変えて正解率を測る.ipynb`

本文のコードをコードセルで直接実行します。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2', 'transformers==5.22.1', 'tiktoken==0.14.0', 'scikit-learn>=1.5,<2', 'pydantic==2.14.2', 'outlines==1.3.3', 'lancedb==0.41.0', 'pyarrow==26.0.1', 'sentence-transformers==6.2.0']], check=True)


In [ ]:
# 7 章：指示の書き方で、小さなモデルの分類の成績がどう変わるか（Qwen2.5-0.5B-Instruct・貪欲デコード）
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 本書で用意した感想文（P＝ポジティブ、N＝ネガティブ）。後半 8 件は否定・皮肉・逆接を含む
DATA = [("届いてすぐ使えて、とても便利です。", "P"), ("説明書が分かりやすく、迷わず設定できた。", "P"),
        ("値段のわりに質感が良くて満足している。", "P"), ("サポートの対応が早く、助かりました。", "P"),
        ("電池の持ちが長く、出張でも安心だ。", "P"), ("音が静かで、夜でも気にならない。", "P"),
        ("二日で壊れた。返品したい。", "N"), ("注文から三週間たっても届かない。", "N"),
        ("画面が暗くて、昼間はほとんど見えない。", "N"), ("問い合わせても返事がまったく来ない。", "N"),
        ("重すぎて持ち運ぶ気にならない。", "N"), ("説明と色が違っていてがっかりした。", "N"),
        ("期待していなかったが、思ったより悪くない。", "P"), ("高いだけあって、文句のつけようがない。", "P"),
        ("最初は戸惑ったが、慣れると手放せない。", "P"), ("不満がないわけではないが、また買うと思う。", "P"),
        ("デザインは素敵だが、肝心の性能が物足りない。", "N"), ("さすが有名メーカー、三日で動かなくなるとは。", "N"),
        ("安かったので仕方ないが、二度と買わない。", "N"), ("見た目は良い。見た目だけは。", "N")]
SHOTS = [("箱を開けた瞬間から気に入りました。", "ポジティブ"), ("一度使っただけで電源が入らなくなった。", "ネガティブ"),
         ("値段は高めだが、それ以上の価値がある。", "ポジティブ")]
SHOTS_BAL = SHOTS + [("説明書どおりにしても動かず、時間を無駄にした。", "ネガティブ")]   # 2 対 2 にそろえる
LABEL = {"P": "ポジティブ", "N": "ネガティブ"}

def prompts(text):
    fmt = "答えは「ポジティブ」か「ネガティブ」の 1 語だけで、ほかには何も書かない。"
    few = "".join(f"文：{s}\n答え：{a}\n\n" for s, a in SHOTS)
    bal = "".join(f"文：{s}\n答え：{a}\n\n" for s, a in SHOTS_BAL)
    return {
        "① 指示だけ":            [{"role": "user", "content": f"次の文の感情を判定して。\n{text}"}],
        "② 形式を指定":          [{"role": "user", "content": f"次の文の感情を判定して。{fmt}\n文：{text}"}],
        "③ 形式＋例を 3 つ":      [{"role": "user", "content": f"次の文の感情を判定して。{fmt}\n\n{few}文：{text}\n答え："}],
        "④ 役割と判断基準も書く": [{"role": "system", "content": "あなたは商品レビューを分類する担当者です。"
                                    "文の後半や逆接（「が」「だけは」）のあとに書き手の結論があることが多いので、結論で判断します。"},
                                   {"role": "user", "content": f"{fmt}\n\n{few}文：{text}\n答え："}],
        "⑤ 形式＋例を 4 つ（2 対 2）": [{"role": "user", "content": f"次の文の感情を判定して。{fmt}\n\n{bal}文：{text}\n答え："}],
    }

name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).eval()

@torch.no_grad()
def answer(messages):
    ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    out = model.generate(**ids, max_new_tokens=12, do_sample=False)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def choose(messages, labels=("ポジティブ", "ネガティブ")):
    """⑥ 生成せずに、各ラベルを続けたときの対数確率を比べて選ぶ（8 章の制約デコーディングと同じ考え方）"""
    base = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"]
    scores = []
    for lab in labels:
        lab_ids = tok(lab, add_special_tokens=False, return_tensors="pt").input_ids
        ids = torch.cat([base, lab_ids], dim=1)
        logp = model(ids).logits[0, base.shape[1] - 1:-1].log_softmax(-1)
        scores.append(logp.gather(1, lab_ids[0][:, None]).sum().item())
    return labels[max(range(len(labels)), key=lambda i: scores[i])]

conds = list(prompts("x")) + ["⑥ ③の文面で、2 つのラベルの確率を比べる"]
results = {k: {"ok": 0, "hard_ok": 0, "fmt_ng": 0, "pos": 0} for k in conds}
raw = []                       # 解析用の出力。掲載用の完全な実行証跡は別に保存する
for i, (text, gold) in enumerate(DATA):
    ps = prompts(text)
    outs = {k: answer(m) for k, m in ps.items()}
    outs[conds[-1]] = choose(ps["③ 形式＋例を 3 つ"])
    raw.append({"text": text, "gold": LABEL[gold], "answers": outs})
    for k, a in outs.items():
        found = [lab for lab in ("ポジティブ", "ネガティブ") if lab in a]
        r = results[k]
        r["fmt_ng"] += a not in ("ポジティブ", "ネガティブ")      # 1 語だけ、を守れなかった
        r["pos"] += found == ["ポジティブ"]
        if len(found) == 1 and found[0] == LABEL[gold]:
            r["ok"] += 1; r["hard_ok"] += i >= 12
for k, r in results.items():
    print(f"{k}：正解 {r['ok']}/20（難しい 8 件で {r['hard_ok']}/8）  形式違反 {r['fmt_ng']}/20  「ポジティブ」と答えた {r['pos']}/20（正解は 10）")
import json
json.dump(raw, open("ch07_prompt_raw.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
